# 00 — Setup and preprocessing

Run **once per Drive**. Cells 1–4 also run at the start of every session,
because Colab discards the compiled extensions.

Order matters: undistortion must precede the dense clouds, because
`roma_init` uses the same scene reader.

Every cell is safe to re-run.


## 1. Drive and paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------------------
# The one place paths are defined. Everything else derives from DRIVE_ROOT.
#
#   e3dgsuw/
#     dataset/     the four scenes (original) + undistorted/  <- created below
#     dense/       M1 clouds, with SHA-256 sidecars
#     runs/        <cell>/<scene>/s<seed>/  -- one run, all of it together
#     analysis/    analyse.py output, figures, tables
#     run_ledger.json
# ---------------------------------------------------------------------------
DRIVE_ROOT   = '/content/drive/MyDrive/e3dgsuw'
DATASET_DIR  = f'{DRIVE_ROOT}/dataset'
DATA_UNDIST  = f'{DATASET_DIR}/undistorted'
DENSE_DIR    = f'{DRIVE_ROOT}/dense'
ANALYSIS_DIR = f'{DRIVE_ROOT}/analysis'

# Training reads from local disk, not Drive: the scene loader pulls every image
# at startup, and Drive's FUSE layer makes that far slower than a single copy.
LOCAL_DATA   = '/content/data'

REPO_URL  = 'https://github.com/dinanirham/An-Efficient-3D-Gaussian-Splatting-for-Underwater-3D-Reconstruction.git'
REPO_DIR  = '/content/e3dgsuw'
IMPL_DIR  = f'{REPO_DIR}/implementation'
SCENES    = ['Curasao', 'IUI3-RedSea', 'JapaneseGradens-RedSea', 'Panama']

import os
assert os.path.isdir(DRIVE_ROOT), (
    f'{DRIVE_ROOT} not found. Check the folder name, or edit DRIVE_ROOT above.')
for d in (DATA_UNDIST, DENSE_DIR, f'{DRIVE_ROOT}/runs', ANALYSIS_DIR):
    os.makedirs(d, exist_ok=True)

# Export them so the `!` cells below resolve "$DRIVE_ROOT" as a real shell
# variable. Relying on IPython to substitute notebook variables into magics
# works until it doesn't, and when it doesn't it substitutes nothing and the
# command runs against a silently truncated path rather than failing.
os.environ.update(
    DRIVE_ROOT=DRIVE_ROOT, DATASET_DIR=DATASET_DIR, DATA_UNDIST=DATA_UNDIST,
    DENSE_DIR=DENSE_DIR, ANALYSIS_DIR=ANALYSIS_DIR, LOCAL_DATA=LOCAL_DATA,
    REPO_DIR=REPO_DIR, IMPL_DIR=IMPL_DIR,
)


def find_originals():
    """Locate the four scenes under dataset/, however they were arranged.

    Accepts the scenes directly under dataset/, or nested one level (e.g.
    dataset/SeathruNeRF_dataset/). Returns the directory that contains them.
    """
    candidates = [DATASET_DIR] + [
        os.path.join(DATASET_DIR, d) for d in sorted(os.listdir(DATASET_DIR))
        if os.path.isdir(os.path.join(DATASET_DIR, d)) and d != 'undistorted'
    ]
    for base in candidates:
        if all(os.path.isdir(os.path.join(base, s)) for s in SCENES):
            return base
    return None


DATA_ORIG = find_originals()

# verify_undistort's T1 -- the check that would catch the undistortion gap --
# reads the *original* dataset. Point it at wherever it actually landed on
# Drive, or T1 reports "dataset not found" and the one check that matters here
# quietly stops testing anything.
if DATA_ORIG:
    os.environ['E3DGSUW_DATASET'] = DATA_ORIG

print('drive root :', DRIVE_ROOT)
print('originals  :', DATA_ORIG or 'NOT FOUND')
print('undistorted:', DATA_UNDIST)


## 2. GPU — must be an A100

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
cap  = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
print(f'torch {torch.__version__}  cuda {torch.version.cuda}  {name}  sm_{cap[0]}{cap[1]}')

# Every conclusion in this study is a between-cell contrast, and cells on
# different devices are not comparable. Stop now rather than produce a run
# that has to be discarded later.
assert 'A100' in name, f'Expected an A100, got {name!r}. Restart the runtime.'


## 3. Clone the repository

In [ ]:
import os, subprocess

# Private repo? Add a Colab secret named GITHUB_TOKEN (key icon in the left
# sidebar) with a fine-grained read token, and toggle notebook access on.
# Read from Secrets rather than pasted into the cell: a pasted token is saved
# inside the .ipynb, which then travels wherever the notebook does.
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') or None
    print('GITHUB_TOKEN: loaded from Colab Secrets')
except ImportError:
    pass                                  # not running under Colab
except Exception as e:                    # secret absent, or access not granted
    print(f'GITHUB_TOKEN: not available ({type(e).__name__}) -- '
          'fine for a public repo')

url = REPO_URL
if GITHUB_TOKEN:
    url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')

# Never let git fall back to an interactive credential prompt: in a notebook it
# hangs the cell indefinitely with nothing on screen to say why.
env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}


def _redact(s):
    """Strip the token from git output -- git echoes the remote URL on failure,
    and notebook outputs are saved to the file and shared with it."""
    return s.replace(GITHUB_TOKEN, '***') if GITHUB_TOKEN else s


if os.path.isdir(REPO_DIR) and not os.path.isdir(f'{REPO_DIR}/.git'):
    raise RuntimeError(
        f'{REPO_DIR} exists but is not a git checkout -- probably a clone that '
        f'died partway. Delete it and re-run this cell.')

if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git','clone','--depth','1',url,REPO_DIR],
                       capture_output=True, text=True, env=env)
    if r.returncode != 0:
        raise RuntimeError(
            'clone failed. If the repository is private, add a GITHUB_TOKEN '
            'secret in Colab and grant this notebook access.\n'
            f'{_redact(r.stderr)[-800:]}')
else:
    # Repoint the remote before pulling. The stored URL was written by an
    # earlier clone, which may have run without a token (or with a stale one);
    # injecting the token into `url` alone never reaches the pull.
    subprocess.run(['git','-C',REPO_DIR,'remote','set-url','origin',url],
                   check=True, env=env)
    r = subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],
                       capture_output=True, text=True, env=env)
    if r.returncode != 0:
        raise RuntimeError(
            'pull failed. If the repository is private, check the GITHUB_TOKEN '
            'secret is set and this notebook has access.\n'
            f'{_redact(r.stderr)[-800:]}')

# Fail here, naming the directory, rather than letting a later cell run from
# whatever the working directory happened to be.
assert os.path.isdir(IMPL_DIR), (
    f'clone produced no {IMPL_DIR}. Contents of {REPO_DIR}: '
    f'{sorted(os.listdir(REPO_DIR)) if os.path.isdir(REPO_DIR) else "missing"}')

os.chdir(IMPL_DIR)
print(subprocess.run(['git','-C',REPO_DIR,'log','--oneline','-1'],
                     capture_output=True, text=True).stdout.strip())
print('cwd:', os.getcwd())


## 4. Build the CUDA extensions  *(a few minutes, every session)*

In [ ]:
# Builds diff_gaussian_rasterization_ms and simple_knn against whatever torch
# Colab ships -- deliberately NOT installing our own, which would risk a
# mismatch between torch's CUDA and the toolkit the extensions compile with.
# Takes a few minutes; must be repeated each session.
#
# chdir explicitly rather than via `%cd $IMPL_DIR`: a magic whose variable fails
# to expand reports the *current* directory and continues, so the build then
# runs from the wrong place and fails two steps later with a bare
# "tools/setup_colab.sh: No such file or directory".
import os
assert os.path.isdir(IMPL_DIR), (
    f'{IMPL_DIR} not found -- run the "Clone the repository" cell above first.')
os.chdir(IMPL_DIR)
print('building in', os.getcwd())
!bash tools/setup_colab.sh


In [ ]:
import importlib, torch
for m in ('diff_gaussian_rasterization_ms', 'simple_knn'):
    importlib.import_module(m)
print('extensions import OK  |  torch', torch.__version__,
      '| cuda', torch.version.cuda)


## 5. Verify the rasterizer merge

The gate. The whole merge rests on one identity: for a single Gaussian at depth
`z` the probe gives `Z_raw = α·z`, so `Z_raw/α` must recover `z` on every
covered pixel. Verified on sm_86 during development — this confirms it on the
A100 before anything is trained on top of it.


In [ ]:
!python -m tools.verify_rasterizer


## 6. The remaining self-checks

Seventy-six checks across ten suites. Cheap, and several encode findings that
are easy to reintroduce.


In [ ]:
for t in ['verify_config_layer','verify_ledger','verify_metrics','verify_storage',
          'verify_analysis','verify_undistort','verify_dense_init','verify_simplify',
          'verify_quantize']:
    !python -m tools.{t} 2>&1 | tail -2


## 7. Locate the dataset

Expects the four scenes under `dataset/` — either directly, or nested one level
(e.g. `dataset/SeathruNeRF_dataset/`). Both layouts are accepted.


In [ ]:
import os
assert DATA_ORIG, (
    f'Could not find the four scenes under {DATASET_DIR}.\n'
    f'Expected {SCENES}\n'
    f'either directly in dataset/ or one level down.\n'
    f'Found: {sorted(os.listdir(DATASET_DIR))}')

for s in SCENES:
    d = [x for x in os.listdir(f'{DATA_ORIG}/{s}') if x.lower() == 'images_wb'][0]
    n = len(os.listdir(f'{DATA_ORIG}/{s}/{d}'))
    print(f'{s:24s} {n:3d} images   dir: {d}')
print('\nExpect 21 / 29 / 20 / 18. Note IUI3-RedSea uses a capital-I Images_wb.')


## 8. COLMAP undistortion — **required**

All four scenes ship with the COLMAP **OPENCV** camera model and real
distortion coefficients, while the scene reader accepts only
PINHOLE/SIMPLE_PINHOLE. Without this step every run fails at scene load.

Idempotent — re-running this notebook will not resample the images again.


In [ ]:
import shutil, subprocess
if shutil.which('colmap') is None:
    !apt-get -qq update > /dev/null 2>&1
    !apt-get -qq install -y colmap > /dev/null 2>&1
assert shutil.which('colmap'), (
    'colmap not installed. Try:  !apt-get install -y colmap\n'
    'Undistortion cannot be skipped -- the scenes are OPENCV-model.')
print(subprocess.run(['colmap','-h'], capture_output=True, text=True).stdout[:150])


In [ ]:
import subprocess, time
t0 = time.time()
for s in SCENES:
    print(f'--- {s} ---', flush=True)
    r = subprocess.run(['python','-m','source.undistort',
                        '--source', f'{DATA_ORIG}/{s}',
                        '--output', f'{DATA_UNDIST}/{s}'],
                       capture_output=True, text=True)
    print((r.stdout or r.stderr)[-700:], flush=True)
    if r.returncode != 0:
        raise RuntimeError(f'undistortion failed for {s}')
print(f'\ntotal {(time.time()-t0)/60:.1f} min')


In [ ]:
# Confirm every scene will now load.
import sys
from pathlib import Path
sys.path.insert(0, IMPL_DIR)
from source.undistort import verify_undistorted
for s in SCENES:
    i = verify_undistorted(Path(DATA_UNDIST) / s)
    n = len(list((Path(DATA_UNDIST) / s / 'images').iterdir()))
    print(f'{s:24s} {i["model"]:16s} {i["width"]}x{i["height"]}  {n} images')


## 9. Dense clouds — for the M1 cells (A1, A4, A5, A7)

One per scene, roughly 10–25 minutes each. The preset is a real experimental
choice: with densification disabled the primitive count can never grow, so a
cloud below the budget makes A4 collapse onto A1 and A7 onto A5. Compare these
counts against the budget once S1 has produced one.

Preprocessing wall-clock is **not** part of training time — report it
alongside, or A1's cost is understated relative to A0's.


In [ ]:
import subprocess, time
N_BUD = 400_000          # configs/cells.json; see ablation_design.md 5
failures = []
for s in SCENES:
    out = f'{DENSE_DIR}/{s}.ply'
    if os.path.exists(out):
        print(f'{s}: already present, skipping'); continue
    print(f'--- {s} ---', flush=True)
    t0 = time.time()
    r = subprocess.run(['python','-m','source.roma_init',
                        '--source_path', f'{DATA_UNDIST}/{s}',
                        '--output',      out,
                        '--images',      'images',
                        '--preset',      'dense',
                        '--seed',        '0'],
                       capture_output=True, text=True)
    print(r.stdout[-900:], flush=True)

    # Check the exit code, and print stderr on its own. `r.stdout or r.stderr`
    # hides the traceback whenever stdout is non-empty -- and it always is
    # here, because the scene reader chatters before anything can fail. That
    # combination reported four crashed runs as four successes.
    if r.returncode != 0:
        print(f'!!! {s} FAILED (exit {r.returncode})', flush=True)
        print(r.stderr[-1500:], flush=True)
        failures.append(s)
    elif not os.path.exists(out):
        print(f'!!! {s} exited 0 but wrote no {out}', flush=True)
        failures.append(s)
    else:
        n = os.path.getsize(out) / 1e6
        print(f'{s}: {(time.time()-t0)/60:.1f} min   {out}  {n:.1f} MB')
        # The budget must lie below this, or M2 is inert under M1 and A4
        # collapses onto A1 (ablation_design.md 5). Preflight refuses such a
        # run, but seeing it here costs nothing and saves a wasted queue entry.
        import json as _json
        _side = out.replace('.ply', '.json')
        if os.path.exists(_side):
            _n = _json.load(open(_side)).get('kept')
            if _n:
                _ok = 'ok' if N_BUD < _n else 'TOO HIGH -- lower n_bud'
                print(f'    points={_n:,}   n_bud={N_BUD:,}   {_ok}')

if failures:
    raise RuntimeError(
        f'dense-cloud generation failed for {failures}. The M1 cells '
        f'(A1/A4/A5/A7) cannot run without these. A0/A2/A3/A6 are unaffected, '
        f'so S1 can still proceed.')
print('\nall dense clouds present')


## 10. Initialise the ledger

96 rows: 8 cells × 4 scenes × 3 seeds. Refuses to overwrite a campaign in
progress unless `--force`.


In [ ]:
!python -m tools.run_ledger init   --output_root "$DRIVE_ROOT"
!python -m tools.run_ledger status --output_root "$DRIVE_ROOT"


---
**Next:** open `01_worker.ipynb` and run it. Repeat every session until the
ledger reports everything done.
